# ML-04 — Data Contract & Schema Specifications

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Data contracts, missingness handling, and feature classification.

## 1. Grain and Time Horizon

- **Unit of Analysis (Grain):** One row per pseudonymized content item (`content_id`).
- **Time Window:** Trailing 90-day aggregate search and engagement metrics.
- **Clients:** 32 distinct anonymized client domains.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect data path: works in local repo or directly in Google Colab
DATA_URL = 'https://raw.githubusercontent.com/AzizullahMemonAi/FlyRank-ML-Assignments/main/data/raw/content_refresh_anonymized.csv'
LOCAL_PATHS = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv')
]
data_path = next((p for p in LOCAL_PATHS if p.exists()), None)
df = pd.read_csv(data_path if data_path is not None else DATA_URL)
print(f'Dataset Grain Verification: {len(df)} rows, {df["content_id"].nunique()} unique content_ids.')
assert len(df) == df['content_id'].nunique(), 'Grain violation detected!'


Dataset Grain Verification: 30,000 rows, 30,000 unique content_ids.


## 2. Field Classification & Exclusions

- **Clustering Feature Candidates:** `log1p(impressions_90d)`, `log1p(clicks_90d)`, `clean_avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, `days_with_impressions`, `log1p(content_age_days)`, `log1p(days_since_last_update)`, `clean_word_count`, `has_valid_position`, `has_keyword_data`.
- **Label-Derived Fields (EXCLUDED):** `trend_direction`, `trend_pct`, `is_declining_label` (excluded to avoid target contamination).
- **Identifiers (EXCLUDED from features):** `content_id`, `client_id` (used strictly for group-level validation splits).
- **Future / Outcome Windows (EXCLUDED):** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`.

In [1]:
excluded_cols = ['trend_direction', 'trend_pct', 'content_id', 'client_id', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
print('Explicitly Excluded Columns:', excluded_cols)


Explicitly Excluded Columns: ['trend_direction', 'trend_pct', 'content_id', 'client_id', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']


## 3. Missingness & Sentinel Value Protocol

1. **`avg_position = 0`**: In Google Search Console telemetry, 0 indicates no rank telemetry (unranked). We create indicator `has_valid_position = (avg_position > 0)` and impute `clean_avg_position = 100.0`.
2. **`search_volume` and `word_count`**: Missingness is structured by `content_type` (feedly articles lack keywords). We add `has_keyword_data` and `has_word_count` flags.
3. **Rate columns**: `ctr`, `engagement_rate`, `scroll_rate` are percentage rates scaled by 100.

In [1]:
pos_zero = (df['avg_position'] == 0).sum()
kw_null = df['search_volume'].isna().sum()
print(f'avg_position == 0 count: {pos_zero:,} ({pos_zero/len(df):.1%})')
print(f'search_volume is NaN count: {kw_null:,} ({kw_null/len(df):.1%})')


avg_position == 0 count: 1,205 (4.0%)
search_volume is NaN count: 2,096 (7.0%)


## Self-check

- [x] Data contract verified with automated assertions
- [x] Label leakage vectors identified and excluded
- [x] Missingness patterns mapped to feature flags